# CRAD lab · Colab

**TwoQuarks Research** · capture and activation patching on Qwen2.5-7B-Instruct (4-bit), with the instrument pinned at `eb8217b`.

The lab code comes straight from GitHub ([Jaime2pb3/CRAD](https://github.com/Jaime2pb3/CRAD), folder `lab/`). Nothing has to be uploaded. Google Drive is optional and only used to keep your own results between sessions.

**The ladder is not skipped.** Each cell is a rung. If a cell ends with `🛑 RED RUNG`, **do not move to the next one**: copy the error.

| Cell | Rung | Where | Approx. cost |
|---|---|---|---|
| 1 | clone the lab from GitHub, utilities, optional Drive | CPU or GPU | ~1 min |
| 1b | lab modules: sha256 against the validated versions, selftests | CPU or GPU | ~3 min |
| 2–3 | dependencies, AST, imports, 7 lab selftests | CPU or GPU | ~1 min |
| 4 | end-to-end mock (capture → contract → dose → patch) | CPU | ~2 min |
| 5 | GPU preflight: real load, exact fp16, reproducibility, hook | GPU | ~5 min + model download |
| 6 | cost estimate with times **measured on this GPU** | — | 0 |
| 7–8 | captures (deterministic reference and patching) | GPU | see cell 6 |
| 9 | contract and **pairs chosen by rule** | GPU | ~1 min |
| 10–11 | `patch_probe` on the anchor pair, `patch_batch` (per layer, resumable) | GPU | see cell 6 |
| 12 | **BELENC**: linear response and Fisher + validation sheets | GPU | ~10 min GPU + CPU |
| 13 | **the how**: `story_copy` and `patch_how` attend + traces | GPU | ~20 min |
| 14 | save and **release the runtime** | — | 0 |

**Rules:**
1. The whole series runs on **the same GPU**. Set `EXPECT_GPU` in cell 0 to stop if the runtime is a different one. Cells 12 and 13 check again against the capture GPU, because the replay gates need token-exact reproduction of the captured turn.
2. With `USE_DRIVE = True`, results go to `results_<SERIES>/` in your Drive. Different series never mix.
3. Pairs are **not written by hand**. At T=0.7 outcomes can split differently on another GPU, so cell 9 picks them from this capture's contract, always with the same history length (`global_token`).
4. Colab does **not** run in the background: if you close the tab, it stops. Cell 14 releases the runtime if `RELEASE_RUNTIME_AT_END = True`.
5. Every lab module is checked by sha256 in cell 1b against the version that passed its selftests before any GPU run.


## 0 · Parameters
The only thing you edit. `REPO_REF` is a branch, tag or commit of the repo. `EXPECT_GPU` is part of the GPU name (for example `"A100"` or `"RTX PRO 6000"`), or `None` to accept any GPU. `UNITS_PER_HOUR` is read in the **Resources** panel and only feeds the cost estimate.

In [ ]:
REPO_URL       = "https://github.com/Jaime2pb3/CRAD"
REPO_REF       = "main"
USE_DRIVE      = False
DRIVE_DIR      = "/content/drive/MyDrive/crad-lab"
SERIES         = "colab"
EXPECT_GPU     = None
LAB            = "/content/CRAD/lab"
CONFIG_REF     = "configs/qwen2.5-7b_det_L14-20.json"
CONFIG_PATCH   = "configs/qwen2.5-7b_T0.7_cal0_L14-20_s42.json"
SUPERSET       = (13, 28)
PATCH_LAYERS   = [14, 17, 19, 21, 23, 25]
PATCH_ALIGN    = ["anchor", "commit"]
UNITS_PER_HOUR = None
RUN_BATCH      = False
RELEASE_RUNTIME_AT_END = False
BELENC_LAYERS  = [14, 17, 21]
BELENC_ROW     = 2
REAL_PAIR      = None
HOW_TRACES     = None

## 1 · Lab from GitHub, utilities and optional Drive
Clones the repo at `REPO_REF` (or updates it if it is already there), prints the exact commit, and if `USE_DRIVE` is on, mounts your Drive and restores results from previous sessions.

In [ ]:
import os, sys, json, glob, re, shutil, subprocess, time
from pathlib import Path

REPO_DIR = Path(LAB).parent
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", "FETCH_HEAD"], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True, capture_output=True).stdout.strip()
assert Path(LAB, "run.py").exists(), f"{LAB}/run.py not found: is REPO_REF a version that has lab/?"
print(f"lab at {LAB} · {REPO_URL} @ {REPO_REF} · commit {COMMIT[:12]}")

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ.pop("CRAD_MOCK_MODEL", None)
OUT_DIRS = ("captures", "runs", "preflight", "analysis")
RESULTS = Path(DRIVE_DIR) / f"results_{SERIES}" if USE_DRIVE else None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS.mkdir(parents=True, exist_ok=True)


def _copy_newer(src, dst):
    n = 0
    for p in Path(src).rglob("*"):
        if p.is_dir():
            continue
        q = Path(dst) / p.relative_to(src)
        if q.exists() and q.stat().st_size == p.stat().st_size and q.stat().st_mtime >= p.stat().st_mtime:
            continue
        q.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, q)
        n += 1
    return n


def sync():
    if not USE_DRIVE:
        print("[sync] USE_DRIVE=False: results stay in", LAB)
        return
    n = sum(_copy_newer(Path(LAB, d), RESULTS / d) for d in OUT_DIRS if Path(LAB, d).exists())
    print(f"[sync] {n} new or changed file(s) → {RESULTS}")


def restore():
    if not USE_DRIVE:
        return
    n = sum(_copy_newer(RESULTS / d, Path(LAB, d)) for d in OUT_DIRS if (RESULTS / d).exists())
    print(f"[restore] {n} file(s) from previous sessions ← {RESULTS}")


def sh(cmd, env=None, ok_codes=(0,)):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=LAB, text=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc not in ok_codes:
        raise RuntimeError(f"🛑 RED RUNG (exit {rc}). Do NOT move to the next cell: copy the error above.")
    return rc


def latest(pattern):
    hits = sorted(glob.glob(str(Path(LAB, pattern))))
    return hits[-1] if hits else None


def cfg_of(path):
    return json.load(open(Path(LAB, path)))


def tag(path):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", cfg_of(path)["name"]).strip("-")


def rel(p):
    return str(Path(p).relative_to(LAB))


restore()
os.chdir(LAB)

_smi = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, text=True, capture_output=True)
GPU_NOW = _smi.stdout.strip().splitlines()[0] if _smi.returncode == 0 and _smi.stdout.strip() else "CPU"
if GPU_NOW != "CPU" and EXPECT_GPU and EXPECT_GPU not in GPU_NOW:
    raise RuntimeError(f"🛑 this series is {EXPECT_GPU} and the runtime has {GPU_NOW}. Change the runtime type.")
print("GPU:", GPU_NOW, "(fine for cells 1-4; from cell 5 on you need a GPU)" if GPU_NOW == "CPU" else "✓")

## 1b · Lab modules: sha256 and selftests
Each module is checked against the hash of the version that passed its selftests. If one does not match (for example because `REPO_REF` points to another version), the cell stops before any GPU time is spent.

In [ ]:
import hashlib
NEW = {"crad_rings.py":     "05426935bc404605e880b9be7b787f4c83c868b58a879a63461fa3195dfd82e4",
       "crad_tube.py":      "e7902e5baf9ac255d78700b2e3c86628252e9735c1ac1863a0624808cc8bacc6",
       "patch_how.py":      "ced090579f45939a4d8a4a2407619d2aaeb0945b50872e854c02087059b4dc37",
       "story_copy.py":     "bb962fcff196bc431e0b9b05551cfa6cd577e42f8ef7bef719ef5e0acfdefb12",
       "belenc.py":         "715d5b8583663a4ffb7dfe0d336c4f0a17dd68cc3715c6900f3c4f88f78b9032",
       "belenc_fisher.py":  "cff94e42518c7bb2ce5ee05b834ad2eaec8150fbb478c4d573d43c0b446f2852"}
for f, h in NEW.items():
    got = hashlib.sha256(Path(LAB, f).read_bytes()).hexdigest()
    if got != h:
        raise RuntimeError(f"🛑 {f}: sha256 {got[:12]} ≠ {h[:12]}. Not the validated version (check REPO_REF).")
    print(f"  {f:18s} {got[:12]}  ok")


def same_gpu_or_stop(cap_dir, what):
    """Replay only reproduces the captured turn on the same GPU as the capture."""
    import torch
    want = (json.load(open(f"{cap_dir}/runtime.json")).get("gpu") or {}).get("name")
    have = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    if want is None or have != want:
        raise RuntimeError(f"🛑 {what} needs the capture GPU ({want}); this runtime has {have}.")
    print(f"GPU {have} == capture ✓")


def need_cap():
    if "cap" not in globals() or not cap or REAL_PAIR is None or HOW_TRACES is None:
        raise RuntimeError("🛑 `cap` or the pairs are missing: run cells 8 and 9 first (patching capture and contract).")


for f in NEW:
    sh(f"python {f} --selftest")
print("\nrung 1b · modules verified and selftests green")

## 2 · Dependencies and environment
Recorded in the fingerprint.

In [ ]:
sh("pip install -q -U bitsandbytes && pip install -q accelerate")
sh('python -c "import torch, transformers, accelerate, bitsandbytes, numpy; '
   'print(\'torch\', torch.__version__, \'| cuda\', torch.version.cuda, \'| transformers\', transformers.__version__, '
   '\'| accelerate\', accelerate.__version__, \'| bitsandbytes\', bitsandbytes.__version__, \'| numpy\', numpy.__version__)"')
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'no GPU: CPU runtime (fine for cells 1-4)'")

## 3 · Rungs 0–2 (CPU): AST, imports, selftests

In [ ]:
import ast
for f in sorted(glob.glob(f"{LAB}/*.py") + glob.glob(f"{LAB}/src/*.py")):
    ast.parse(open(f).read(), f)
print("rung 0 · AST OK")
sh('python -c "import run, fingerprint, launch_common, crad_launch, capture_launch, capture, crad_contract, '
   'crad_triad, CRAD_v1_5_prismatic, dose_probe, patch_probe, patch_batch, preflight_gpu, mock_ladder; print(\'rung 1 · import OK\')"')
for c in ("src/CRAD_v1.3.py --selftest", "src/crad_base.py --selftest", "CRAD_v1_5_prismatic.py --selftest",
          "crad_triad.py --selftest", "crad_contract.py selftest", "run.py --selftest", "capture_launch.py --selftest"):
    sh(f"python {c}")
print("\nrung 2 · the 7 selftests are green")

## 4 · Rung 3 (CPU): the whole lab on a fake model
Runs in a temporary copy and does not touch `captures/`. The numbers only check the gates and the exit codes.

In [ ]:
sh("python mock_ladder.py", env={"CRAD_MOCK_MODEL": "1"})

## 5 · Rung 4 (GPU): preflight
From here on we use the GPU. Checks: no offload, correct commit, fp16 hidden states, token+phi reproducibility, and that the self-patch hook writes where phi is read.

In [ ]:
if GPU_NOW == "CPU":
    raise RuntimeError("🛑 this runtime has no GPU: change the runtime type to a GPU and run again from cell 1")
sh(f"python preflight_gpu.py {CONFIG_PATCH}")
sync()

## 6 · Cost estimate with measured times
No projections: we use the seconds per turn that cell 5 measured on this GPU.

In [ ]:
pf = json.load(open(latest("preflight/*.json")))
SPT = pf["sec_per_traced_turn"] * max(1.0, pf["n_tokens"] / max(pf["tokens_in_timed_turn"], 1)) * 1.3
LOAD_S = pf["load_seconds"] + 30
GPU = pf["runtime"]["gpu"].get("name")
if EXPECT_GPU and EXPECT_GPU not in (GPU or ""):
    raise RuntimeError(f"🛑 the latest preflight is from {GPU}, not {EXPECT_GPU}: run cell 5 on that GPU.")


def capture_turns(path):
    a = cfg_of(path)["args"]
    base = 0.0 if a["temperature-mode"] == "off" else a["temp"]
    ct = base if a["calibration-temp"] is None else a["calibration-temp"]
    at = base if a["analysis-temp"] is None else a["analysis-temp"]
    return (1 if ct == 0 else a["calibration-draws"]) * 3 * 8 + (1 if at == 0 else a["draws"]) * 8


def batch_turns_one_layer(n_truth, n_fic, n_align=len(PATCH_ALIGN)):
    pairs, targets = 2 * n_truth * n_fic, n_truth + n_fic
    return targets * 7 + targets + targets * n_align + pairs * n_align * 5


def cost(turns, loads=1):
    h = (turns * SPT + loads * LOAD_S) / 3600
    return f"{turns:>6} turns ≈ {h:5.2f} h" + (f" ≈ {h * UNITS_PER_HOUR:6.1f} units" if UNITS_PER_HOUR else "")


print(f"GPU {GPU} · {pf['tokens_per_second']} tok/s · {SPT:.1f} s/turn (ceiling) · load {LOAD_S:.0f} s\n")
print("reference capture :", cost(capture_turns(CONFIG_REF)))
print("patching capture  :", cost(capture_turns(CONFIG_PATCH)))
for nt, nf in ((3, 3), (5, 5), (8, 8)):
    t = batch_turns_one_layer(nt, nf) * len(PATCH_LAYERS)
    print(f"patch_batch {nt}T×{nf}F, {len(PATCH_LAYERS)} layers:", cost(t, loads=len(PATCH_LAYERS)))

## 7 · Reference capture (deterministic) · cross-hardware replication
Same case as the CRAD preprint. If it matches the previous records (RTX 3060, A100, RTX PRO 6000), the instrument replicates on this hardware too. If not, **it is a provenance finding** (GPU or libraries).

In [ ]:
pat_ref = f"captures/*_{tag(CONFIG_REF)}_cap{SUPERSET[0]}-{SUPERSET[1]}"
cap_ref = latest(pat_ref)
if cap_ref and json.load(open(f"{cap_ref}/capture.json"))["problems"] == []:
    print("an OK capture already exists:", rel(cap_ref))
else:
    sh(f"python capture.py {CONFIG_REF} --superset {SUPERSET[0]} {SUPERSET[1]}")
    sync()
    cap_ref = latest(pat_ref)
    if not cap_ref or json.load(open(f"{cap_ref}/capture.json"))["problems"]:
        raise RuntimeError("🛑 the reference capture is not OK: check CAPTURE.md")

s = json.load(open(f"{cap_ref}/summary.json"))
g, d0 = s["geometry"], s["draws"][0]
ov = [f"{c['global_token']} {c['token']!r}" for c in d0["candidates"] if c["classification"] == "OVERRIDE"]
rows = [("median cos(L,R)", "0.1524", "0.1523", "0.1523", f"{g['cos_LR_median']:.4f}"),
        ("coactivation", "38.8%", "38.9%", "38.9%", f"{100 * g['coactivation_rate_all_states']:.1f}%"),
        ("event token (full leak)", "220", "220", "220", str(d0["event_global_token"])),
        ("OVERRIDE includes", "216 ' is'", "216 ' is'", "216 ' is'", (", ".join(ov[:8]) + (f" (+{len(ov) - 8} more)" if len(ov) > 8 else "")) or "—")]
print(f"{'':34} {'preprint':>10} {'3060 24-Sep':>12} {'A100 22-Oct':>12}  this run")
for r in rows:
    print(f"{r[0]:34} {r[1]:>10} {r[2]:>12} {r[3]:>12}  {r[4]}")
print("\nIf it matches, the instrument replicates on this hardware too. "
      "If not, it is NOT a notebook failure: it is a provenance finding and gets recorded as is.")

## 8 · Patching capture
Deterministic calibration (a single ruler) and 20 draws at T=0.7. `capture.py` checks at value level that what was saved reproduces what CRAD computed. Getting `INCONSISTENT` stops the flow.

In [ ]:
pat = f"captures/*_{tag(CONFIG_PATCH)}_cap{SUPERSET[0]}-{SUPERSET[1]}"
cap = latest(pat)
if cap and json.load(open(f"{cap}/capture.json"))["problems"] == []:
    print("an OK capture already exists:", rel(cap))
else:
    sh(f"python capture.py {CONFIG_PATCH} --superset {SUPERSET[0]} {SUPERSET[1]}")
    sync()
    cap = latest(pat)
    if not cap or json.load(open(f"{cap}/capture.json"))["problems"]:
        raise RuntimeError("🛑 the patching capture is not OK: check CAPTURE.md")
print(open(f"{cap}/CAPTURE.md").read()[:2500])

## 9 · Contract: one rule, one clock, one decision point
Classifies each draw at the decision row (TRUTH / SELF_FICTION / FABRICATE / …).

In [ ]:
sh(f"python crad_contract.py check {rel(cap)}")
sync()
cj = json.load(open(f"{cap}/contract/CONTRACT.json"))
truth = [d["draw"] for d in cj["decisions"] if d["outcome"] == "TRUTH" and d["commit_row"] is not None]
fic = [d["draw"] for d in cj["decisions"] if d["outcome"] == "SELF_FICTION" and d["commit_row"] is not None]
print(f"\ncalibration_id {cj['calibration_id']} · outcomes {cj['outcomes']}")
print(f"TRUTH {truth} · SELF_FICTION {fic}")
if not truth or not fic:
    raise RuntimeError("🛑 There is not at least one TRUTH and one SELF_FICTION: no pair to patch. "
                       "Do not force anything (another seed or config = another series).")


def fmt(tok):
    return "quote" if '"' in tok else ("space" if tok.strip() == "" else "direct")


dec = [d for d in cj["decisions"] if d["outcome"] in ("TRUTH", "SELF_FICTION") and d["row"] is not None]


def pick(src_fmt, tgt_fmt):
    """First TRUTH with format src_fmt and first SELF_FICTION with format tgt_fmt at the SAME global_token."""
    for a in sorted(dec, key=lambda d: d["draw"]):
        if a["outcome"] != "TRUTH" or fmt(a["token"]) != src_fmt:
            continue
        for b in sorted(dec, key=lambda d: d["draw"]):
            if b["outcome"] == "SELF_FICTION" and fmt(b["token"]) == tgt_fmt and b["global_token"] == a["global_token"]:
                return a["draw"], b["draw"]
    return None


anchor_pair = pick("direct", "quote")
commit_pair = pick("quote", "quote")
REAL_PAIR = anchor_pair or commit_pair
HOW_TRACES = [(p_[0], p_[1], al) for p_, al in ((anchor_pair, "anchor"), (commit_pair, "commit")) if p_]

print("\nanchor row format:")
for d in sorted(dec, key=lambda d: d["draw"]):
    print(f"  draw {d['draw']:>2} {d['outcome']:12} {d['token']!r:10} {fmt(d['token']):6} global {d['global_token']}")
print(f"\nanchor pair {anchor_pair} · commit pair {commit_pair}")
print(f"REAL_PAIR = {REAL_PAIR} · HOW_TRACES = {HOW_TRACES}")
if REAL_PAIR is None:
    raise RuntimeError("🛑 there is no TRUTH/SELF_FICTION pair with the same history. Do not force a pair.")
if anchor_pair is None or commit_pair is None:
    print("⚠ one of the two pairs is missing: the how will run only with the one that exists. Note it in the log.")

## 10 · `patch_probe`: the anchor pair (TRUTH → SELF_FICTION, same history)
This is a probe, not a test. Gates: the unpatched run reproduces the captured turn and the self-patch does not move the logits.

In [ ]:
src_d, tgt_d = REAL_PAIR
done = latest(f"{cap}/patch_probe/*_src{src_d}_tgt{tgt_d}/PATCH.md")
if done:
    print("already exists:", rel(done))
else:
    sh(f"python patch_probe.py {rel(cap)} --source {src_d} --target {tgt_d} --layers {' '.join(map(str, PATCH_LAYERS))}")
    sync()
    done = latest(f"{cap}/patch_probe/*_src{src_d}_tgt{tgt_d}/PATCH.md")
print(open(done).read())

## 11 · `patch_batch` per layer (resumable)
Each layer is its own run (the table is already per layer and alignment). Needs ≥3 TRUTH and ≥3 SELF_FICTION.

In [ ]:
def batch_done(layer):
    for p in glob.glob(f"{cap}/patch_batch/*/summary.json"):
        m = json.load(open(p))
        if m["layers"] == [layer] and sorted(m["align"]) == sorted(PATCH_ALIGN):
            return p
    return None


todo = [l for l in PATCH_LAYERS if not batch_done(l)]
print(f"TRUTH {len(truth)} · SELF_FICTION {len(fic)} · pending layers {todo}")
print("estimate:", cost(batch_turns_one_layer(len(truth), len(fic)) * len(todo), loads=len(todo)))
if len(truth) < 3 or len(fic) < 3:
    raise RuntimeError("🛑 patch_batch needs ≥3 TRUTH and ≥3 SELF_FICTION (leave-one-pair-out). Stay with patch_probe.")
if not RUN_BATCH:
    print("\nRUN_BATCH=False → not running. If the estimate works for you: set RUN_BATCH=True in cell 0, run it and come back here.")
else:
    for l in todo:
        sh(f"python patch_batch.py {rel(cap)} --layers {l} --align {' '.join(PATCH_ALIGN)}")
        sync()
    for l in PATCH_LAYERS:
        p = batch_done(l)
        if p:
            print(open(Path(p).with_name("BATCH.md")).read()[:6000], "\n")

## 12 · BELENC: a standard for unknown spatial locations
Physics first, on the capture GPU: linear response with the Fisher metric (`belenc_fisher.py`). It gives the behavioral unit in nats, the behavioral temperature τ, the enrichment E, the latent fragility, χ of the L/R axes against random directions, and the linearity of the real pair. Then the standard, on CPU (`belenc.py`): analytes planted in the real `phi` and the validation sheet of each instrument (FPR, LOD95, random matrix theory), with the LODs converted to nats.

Predictions F1–F4 are written in the header of `belenc_fisher.py`, before running. Resumable: if there are results with the same parameters, they are reused.

In [ ]:
need_cap()
same_gpu_or_stop(cap, "BELENC-Fisher")


def fisher_done():
    for p in sorted(glob.glob(f"{cap}/belenc_fisher/*/fisher.json")):
        m = json.load(open(p))
        if m.get("pair") == list(REAL_PAIR) and sorted(m["layers"]) == sorted(map(str, BELENC_LAYERS)):
            return p
    return None


fj = fisher_done()
if fj:
    print("already exists:", rel(fj))
else:
    sh(f"python belenc_fisher.py {rel(cap)} --layers {' '.join(map(str, BELENC_LAYERS))} "
       f"--pair {REAL_PAIR[0]} {REAL_PAIR[1]}")
    sync()
    fj = fisher_done()
print(open(Path(fj).with_name("FISHER.md")).read())

for L in BELENC_LAYERS:
    if glob.glob(f"{cap}/belenc/*_L{L}_T7_r{BELENC_ROW}/BELENC.md"):
        print(f"the L{L} sheet already exists")
        continue
    sh(f"python belenc.py run {rel(cap)} --layer {L} --row {BELENC_ROW} --real-pair {REAL_PAIR[0]} {REAL_PAIR[1]} "
       f"--instruments oracle pca tube_fork tube_anticip rings --fisher-json {rel(fj)}")
    sync()
for L in BELENC_LAYERS:
    p = sorted(glob.glob(f"{cap}/belenc/*_L{L}_T7_r{BELENC_ROW}/BELENC.md"))[-1]
    print(open(p).read()[:8000], "\n")

## 13 · The how: two competing copies
The system says literally `The access code is {secret}.` and the user seeds `...the access code was` in turn 6. At the anchor row there are two copy sources: exact induction from the system (TRUTH) or approximate induction from the model's own story (SELF_FICTION).

`story_copy` (CPU, seconds) checks whether the fiction copies exactly the code in the `access code was` slot. `patch_how attend` measures which region each head attends to at the decision row, with an FWER null over the 784 heads. `patch_how trace` patches block, attention, MLP and head by head in L0–L26 for each pair in `HOW_TRACES`. Predictions A1–A3 and B1–B3 are in the header of `patch_how.py`. **Read it after the BELENC sheets.**

In [ ]:
need_cap()
sh(f"python story_copy.py {rel(cap)}")
sync()
print(open(f"{cap}/story_copy/STORY_COPY.md").read())

same_gpu_or_stop(cap, "patch_how")
if not glob.glob(f"{cap}/patch_how/*_attend/HOW_ATTEND.md"):
    sh(f"python patch_how.py {rel(cap)} attend")
    sync()
for s_, t_, al in HOW_TRACES:
    if glob.glob(f"{cap}/patch_how/*_trace_src{s_}_tgt{t_}_{al}/HOW_TRACE.md"):
        print(f"trace {s_} → {t_} ({al}) already exists")
        continue
    sh(f"python patch_how.py {rel(cap)} trace --source {s_} --target {t_} --align {al}")
    sync()
for p in sorted(glob.glob(f"{cap}/patch_how/*/HOW_*.md")):
    print(open(p).read()[:6000], "\n")

## 14 · Save and release the runtime
`sync()` always runs (it only copies when `USE_DRIVE` is on). `runtime.unassign()` wipes Colab's local disk: without Drive, everything in `captures/` is lost. That is why it only runs if `RELEASE_RUNTIME_AT_END = True`.

In [ ]:
sync()
print("results at", RESULTS or LAB)
if RELEASE_RUNTIME_AT_END:
    from google.colab import runtime
    print("releasing the runtime…")
    runtime.unassign()
else:
    print("RELEASE_RUNTIME_AT_END=False → the runtime stays connected (and keeps spending units).")